Farmer Decision Support

This notebook demonstrates a machine learning pipeline to predict onion prices and provide decision support for farmers based on these predictions. It includes steps for loading models and data, performing predictions, and analyzing potential profit.


### 1. Import Libraries

This cell imports necessary Python libraries: `pandas` for data manipulation, `numpy` for numerical operations, `tensorflow.keras.models` for loading the pre-trained neural network model, and `joblib` for loading the data scaler. It also prints a confirmation message upon successful import.

In [1]:
import pandas as pd
import numpy as np
print("✅ Libraries imported!")

✅ Libraries imported!


### 2. Upload Model and Scaler Files

This cell uses `google.colab.files` to allow the user to upload the pre-trained `onion_model.h5` (the prediction model) and `onion_scaler.pkl` (the data scaler) to the Colab environment. These files are essential for making predictions.

In [3]:
from google.colab import files

print('Please select the files to upload (e.g., onion_model.h5, onion_scaler.pkl):')
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Please select the files to upload (e.g., onion_model.h5, onion_scaler.pkl):


Saving onion_model.h5 to onion_model.h5
Saving onion_scaler.pkl to onion_scaler.pkl
User uploaded file "onion_model.h5" with length 392616 bytes
User uploaded file "onion_scaler.pkl" with length 719 bytes


### 3. Load Pre-trained Model and Scaler

Here, the uploaded `onion_model.h5` and `onion_scaler.pkl` files are loaded into the Colab session. The `load_model` function from Keras is used for the neural network, and `joblib.load` is used for the scaler. The `compile=False` argument is used for `load_model` as we are only using the model for inference, not training.

In [4]:
from tensorflow.keras.models import load_model
import joblib

onion_model = load_model('onion_model.h5', compile=False)
onion_scaler = joblib.load('onion_scaler.pkl')

print("✅ Onion model loaded!")

✅ Onion model loaded!


### 4. Upload and Process Historical Price Data

This cell prompts the user to upload the `Karnataka_Processed.csv` file, which contains historical onion price data. After uploading, it reads the CSV into a pandas DataFrame. It then converts the 'Price Date' column to datetime objects, ensuring correct format parsing, and displays the shape of the DataFrame to confirm successful loading and processing.

In [7]:
from google.colab import files
import io

uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[file_name]))

df['Price Date'] = pd.to_datetime(df['Price Date'], format='%d/%m/%Y')

print(df.shape)

Saving Karnataka_Processed.csv to Karnataka_Processed (2).csv
(12774, 10)


### 5. Predict Tomorrow's Onion Price

This is the core prediction logic. It first filters the loaded dataset (`df`) to create `onion_df` containing only 'Onion' commodity data. It then extracts the last `LOOKBACK` (30) days of 'Modal_Price', scales this data using the loaded `onion_scaler`, reshapes it for input into the LSTM model, and finally makes a prediction for the next day's price. The predicted price is then inverse-transformed back to the original price scale and displayed alongside today's price.

In [9]:
LOOKBACK = 30

# Filter the main DataFrame for 'Onion' commodity to create onion_df
onion_df = df[df['Commodity'] == 'Onion'].copy()

# Get last 30 days
last_30 = onion_df['Modal_Price'].values[-LOOKBACK:]

# Scale
scaled = onion_scaler.transform(last_30.reshape(-1,1))

# Reshape for LSTM
X_input = scaled.reshape(1, LOOKBACK, 1)

# Predict
pred_scaled = onion_model.predict(X_input, verbose=0)

# Convert back to original price
predicted_price = onion_scaler.inverse_transform(pred_scaled)[0][0]

today_price = onion_df['Modal_Price'].iloc[-1]

print(f"Today's Price     : ₹{today_price:.2f}")
print(f"Predicted Tomorrow: ₹{predicted_price:.2f}")

Today's Price     : ₹2200.00
Predicted Tomorrow: ₹2058.01


### 6. AI Decision Support Result

This cell provides a recommendation based on the predicted price change. It calculates the absolute and percentage difference between today's price and the predicted tomorrow's price. Based on these differences, it gives a recommendation (WAIT, SELL NOW, or HOLD) and a reason, indicating whether the price is expected to increase, decrease, or remain stable.

In [10]:
difference = predicted_price - today_price

if difference > 50:
    recommendation = "WAIT"
    reason = "Price is expected to increase."

elif difference < -50:
    recommendation = "SELL NOW"
    reason = "Price is expected to decrease."

else:
    recommendation = "SELL"
    reason = "No significant price change expected."

print("========== DECISION ==========")
print(f"Recommendation : {recommendation}")
print(f"Expected Change: ₹{difference:.2f}")
print(f"Reason         : {reason}")

========== DECISION ==========
Recommendation : SELL NOW
Expected Change: ₹-141.99
Reason         : Price is expected to decrease.


In [11]:
difference = predicted_price - today_price
percent_change = (difference / today_price) * 100

if difference > 50:
    recommendation = "WAIT"
    reason = "Price is expected to increase tomorrow."

elif difference < -50:
    recommendation = "SELL NOW"
    reason = "Price is expected to decrease tomorrow."

else:
    recommendation = "HOLD"
    reason = "No significant price change expected."

print("=" * 50)
print("AI DECISION SUPPORT RESULT")
print("=" * 50)

print(f"Today's Price        : ₹{today_price:.2f}")
print(f"Predicted Price      : ₹{predicted_price:.2f}")
print(f"Expected Change      : ₹{difference:.2f}")
print(f"Percentage Change    : {percent_change:.2f}%")
print(f"Recommendation       : {recommendation}")
print(f"Reason               : {reason}")

AI DECISION SUPPORT RESULT
Today's Price        : ₹2200.00
Predicted Price      : ₹2058.01
Expected Change      : ₹-141.99
Percentage Change    : -6.45%
Recommendation       : SELL NOW
Reason               : Price is expected to decrease tomorrow.


### 7. Farmer Profit Analysis

This cell allows for a hypothetical profit analysis from a farmer's perspective. The `quantity` and `production_cost` can be adjusted. It calculates the potential revenue and profit for selling today versus selling tomorrow based on the predicted price. It then shows the profit difference and provides a final recommendation (WAIT, SELL NOW, or HOLD) based on whether waiting is expected to increase or decrease profit.

In [12]:
# Farmer Inputs
quantity = 100          # Quintals
production_cost = 1800  # ₹ per Quintal

today_revenue = today_price * quantity
tomorrow_revenue = predicted_price * quantity

today_profit = (today_price - production_cost) * quantity
tomorrow_profit = (predicted_price - production_cost) * quantity

profit_difference = tomorrow_profit - today_profit

print("=" * 55)
print("FARMER PROFIT ANALYSIS")
print("=" * 55)

print(f"Quantity             : {quantity} Quintals")
print(f"Production Cost      : ₹{production_cost:.2f}/Quintal")

print(f"\nToday's Revenue      : ₹{today_revenue:,.2f}")
print(f"Tomorrow's Revenue   : ₹{tomorrow_revenue:,.2f}")

print(f"\nToday's Profit       : ₹{today_profit:,.2f}")
print(f"Tomorrow's Profit    : ₹{tomorrow_profit:,.2f}")

print(f"\nProfit Difference    : ₹{profit_difference:,.2f}")

if profit_difference > 0:
    print("\nRecommendation: WAIT")
    print("Reason: Waiting may increase your profit.")

elif profit_difference < 0:
    print("\nRecommendation: SELL NOW")
    print("Reason: Waiting may reduce your profit.")

else:
    print("\nRecommendation: HOLD")
    print("Reason: Profit remains almost unchanged.")

FARMER PROFIT ANALYSIS
Quantity             : 100 Quintals
Production Cost      : ₹1800.00/Quintal

Today's Revenue      : ₹220,000.00
Tomorrow's Revenue   : ₹205,800.91

Today's Profit       : ₹40,000.00
Tomorrow's Profit    : ₹25,800.90

Profit Difference    : ₹-14,199.10

Recommendation: SELL NOW
Reason: Waiting may reduce your profit.
